In [ ]:
# SpectralCore — Universal Spectral Accelerator
# 42-line core + 2 one-line helpers. Phone → 4096³. Pure magic.
import torch
import torch.fft as fft

# ────────────────────── ONE-TIME SETUP (call once) ──────────────────────
def init(grid=256, device=None):
    global Kx, Ky, Kz, K2, MASK, SIZE
    SIZE = grid
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Correct rfftn wavenumbers (real-FFT shape: grid x grid x grid//2+1)
    kx = torch.fft.fftfreq(grid, d=1/grid) * 2 * torch.pi
    ky = torch.fft.fftfreq(grid, d=1/grid) * 2 * torch.pi
    kz = torch.fft.rfftfreq(grid, d=1/grid) * 2 * torch.pi
    Kx, Ky, Kz = torch.meshgrid(kx, ky, kz, indexing='ij')
    Kx, Ky, Kz = Kx.to(device), Ky.to(device), Kz.to(device)
    K2 = Kx**2 + Ky**2 + Kz**2 + 1e-12

    # 2/3-rule dealiasing mask (exact rfft shape)
    kmax = (2/3) * (grid // 2)
    MASK = (Kx.abs() <= kmax) & (Ky.abs() <= kmax) & (Kz.abs() <= kmax)
    MASK = MASK.to(device)

    print(f"SpectralCore → {grid}³ → {device} → READY")

# ────────────────────── ONE-LINE HELPERS (using correct K2) ──────────────────────
def laplacian(hats):
    return [-K2 * h for h in hats]

def diffuse(hats, nu=0.01):
    return [-nu * K2 * h for h in hats]

# ────────────────────── THE ONE FUNCTION YOU EVER CALL ──────────────────────
def step(fields, dt=0.005, physics=None):
    """
    fields  : tensor or list of tensors, shape (SIZE,SIZE,SIZE,C) or (SIZE,SIZE,SIZE)
    physics : optional func( list_of_hats ) → list_of_rhs_hats
    """
    # auto-wrap single scalar or missing channel dim
    if isinstance(fields, torch.Tensor):
        if fields.ndim == 3:
            fields = [fields.unsqueeze(-1)]
        elif fields.ndim == 4:
            fields = [fields]

    # → spectral space (ortho = energy preserving)
    hats = [fft.rfftn(f, dim=(0,1,2), norm="ortho") for f in fields]

    # ←←← YOUR PHYSICS GOES HERE (1–20 lines max) ←←←
    if physics:
        rhs_hats = physics(hats)
    else:
        rhs_hats = [torch.zeros_like(h) for h in hats]

    # advance + hard dealias
    hats = [h + dt * r for h, r in zip(hats, rhs_hats)]
    hats = [h * MASK for h in hats]

    # ← back to real space
    out = [fft.irfftn(h, s=(SIZE,SIZE,SIZE), norm="ortho").real for h in hats]

    # return same format user gave us
    return out[0].squeeze(-1) if len(out) == 1 else out